# Regression Decision Tree From Scratch
### Using only Python + Pandas (no sklearn, no AI libs)

**Dataset:** House prices — predict sale price (in $k) from size, bedrooms, age, and distance from city center.

**Key idea:** Same tree structure as classification, but:
- Leaves predict a **number** (mean of target values) instead of a class
- Splits use **Variance Reduction** instead of Entropy / Gini

---
## Step 1 — Load & Explore the Dataset

In [1]:
import pandas as pd
import math
from collections import defaultdict

df = pd.read_csv('house_prices.csv')
print(f'Shape: {df.shape}  →  {df.shape[0]} rows, {df.shape[1]} columns')
df

Shape: (25, 5)  →  25 rows, 5 columns


,size_sqft,bedrooms,age_years,distance_km,price_k
0,750,1,30,12,145
1,820,1,25,10,158
2,900,2,20,8,172
3,980,2,15,9,185
4,1050,2,10,7,198
5,1100,2,8,6,210
6,1150,3,12,5,225
7,1200,3,5,4,238
8,1300,3,18,11,220
9,1350,3,3,3,260


In [2]:
print('Target (price_k) statistics:')
print(df['price_k'].describe().round(2))
print()
print('Feature ranges:')
for col in ['size_sqft','bedrooms','age_years','distance_km']:
    print(f'  {col:<15}: {df[col].min()} → {df[col].max()}')

Target (price_k) statistics:
count     25.00
mean     290.12
std       97.27
min      145.00
25%      220.00
50%      275.00
75%      368.00
max      510.00
Name: price_k, dtype: float64

Feature ranges:
  size_sqft      : 750 → 2800
  bedrooms       : 1 → 6
  age_years      : 1 → 30
  distance_km    : 1 → 12


---
## Step 2 — Variance: Measuring Spread

**Variance** is the regression equivalent of Entropy. It measures how spread out the target values are inside a node.

$$Var(S) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \bar{y})^2$$

- Variance = **0** → all target values are identical (perfect node, no split needed)
- High variance → values are all over the place (node needs splitting)

In [16]:
def variance(series):
    return series.var(ddof=0) if len(series) >= 2 else 0.0

# Demonstrate on the full dataset
print(f'Variance of price_k (whole dataset): {variance(df["price_k"]):.2f}')
print(f'Std deviation of price_k           : {df["price_k"].std():.2f}k')
print()

# Pure node example
pure = pd.Series([250, 250, 250, 250])
mixed = pd.Series([150, 250, 350, 450])
print(f'Variance of [250,250,250,250]: {variance(pure):.2f}  ← zero, perfectly pure')
print(f'Variance of [150,250,350,450]: {variance(mixed):.2f}  ← high, very spread out')

Variance of price_k (whole dataset): 9083.79
Std deviation of price_k           : 97.27k

Variance of [250,250,250,250]: 0.00  ← zero, perfectly pure
Variance of [150,250,350,450]: 12500.00  ← high, very spread out


---
## Step 3 — Variance Reduction: Choosing the Best Split

**Variance Reduction (VR)** = how much a split reduces the spread of target values.

$$VR(S, A, t) = Var(S) - \left( \frac{|S_{left}|}{|S|} \cdot Var(S_{left}) + \frac{|S_{right}|}{|S|} \cdot Var(S_{right}) \right)$$

For **continuous features** (like size_sqft), we try every possible midpoint between sorted unique values and pick the threshold with the highest VR.

In [22]:
def variance_reduction(df, feature, target, threshold):
    """VR from splitting `feature` at `threshold`."""
    left  = df[df[feature] <= threshold][target]
    right = df[df[feature] >  threshold][target]

    n = len(df)
    weighted_child_var = (len(left)/n) * variance(left) + (len(right)/n) * variance(right)
    return variance(df[target]) - weighted_child_var


def best_split(df, feature, target):
    """Find threshold with highest variance reduction for a continuous feature."""
    vals = df[feature].sort_values().unique()
    thresholds = [(vals[i] + vals[i+1]) / 2 for i in range(len(vals) - 1)]
    best_vr, best_thresh = -1, None
    for t in thresholds:
        vr = variance_reduction(df, feature, target, t)
        if vr > best_vr:
            best_vr, best_thresh = vr, t

    return best_thresh, best_vr


# Check each feature
features = ['size_sqft', 'bedrooms', 'age_years', 'distance_km']
print('Best split and VR for each feature:')
print('-' * 55)
for f in features:
    thresh, vr = best_split(df, f, 'price_k')
    bar = '█' * int(vr / 100)
    print(f'  {f:<15}  threshold={thresh:<8}  VR={vr:>8.2f}  {bar}')
print()
print('→ Feature with highest VR becomes the root node split.')

Best split and VR for each feature:
-------------------------------------------------------
  size_sqft        threshold=1750.0    VR= 6497.61  ████████████████████████████████████████████████████████████████
  bedrooms         threshold=4.5       VR= 6218.32  ██████████████████████████████████████████████████████████████
  age_years        threshold=6.5       VR= 5147.21  ███████████████████████████████████████████████████
  distance_km      threshold=4.5       VR= 5147.21  ███████████████████████████████████████████████████

→ Feature with highest VR becomes the root node split.


---
## Step 4 — Node Structure

Identical to classification — same `Node` class, same tree shape.
The only difference is that `prediction` now holds a **number** (the mean), not a class label.

In [23]:
class Node:
    def __init__(self):
        self.feature        = None    # which feature to split on
        self.threshold      = None    # the split value (e.g. size_sqft <= 1375)
        self.children       = {}      # {'left': Node, 'right': Node}
        self.prediction     = None    # mean of target at this leaf
        self.is_leaf        = False
        self.n_samples      = 0       # number of samples that reached this node
        self.mean           = 0.0     # mean target value at this node
        self.var            = 0.0     # variance at this node
        self.var_reduction  = 0.0     # VR used for this split (for feature importance)

    def __repr__(self):
        if self.is_leaf:
            return f'Leaf(predict={self.prediction:.1f}k, n={self.n_samples})'
        return f'Node(split: {self.feature} <= {self.threshold}, n={self.n_samples})'

print('Node class ready.')
n = Node()
n.is_leaf = True
n.prediction = 245.5
n.n_samples = 8
print('Sample leaf:', n)

Node class ready.
Sample leaf: Leaf(predict=245.5k, n=8)


---
## Step 5 — Build the Regression Tree

Same recursive logic as ID3, with three stopping conditions:

```
build_tree(data, features):
  ├─ If max_depth reached           → Leaf (predict mean)
  ├─ If fewer than min_samples      → Leaf (predict mean)
  ├─ If variance is already ~zero   → Leaf (predict mean)
  └─ Otherwise:
       1. For each feature, find best threshold by VR
       2. Pick feature+threshold with highest VR
       3. Split into left (<=) and right (>) subsets
       4. Recursively build left and right subtrees
```

`max_depth` and `min_samples` are critical for regression trees — without them the tree will create one leaf per sample and perfectly memorise the training data (overfitting).

In [6]:
def build_tree(df, features, target, max_depth=4, min_samples=3, depth=0):
    node = Node()
    node.n_samples = len(df)
    node.mean      = df[target].mean()
    node.var       = variance(df[target])

    # BASE CASE: stopping conditions
    if depth >= max_depth or len(df) < min_samples or node.var < 1e-6:
        node.is_leaf    = True
        node.prediction = round(node.mean, 2)
        return node

    # Find the best feature and threshold
    best_vr, best_feat, best_thresh = -1, None, None

    for f in features:
        thresh, vr = best_split(df, f, target)
        if thresh is not None and vr > best_vr:
            best_vr, best_feat, best_thresh = vr, f, thresh

    # No useful split found
    if best_vr <= 0 or best_feat is None:
        node.is_leaf    = True
        node.prediction = round(node.mean, 2)
        return node

    # Apply the split
    node.feature       = best_feat
    node.threshold     = round(best_thresh, 2)
    node.var_reduction = best_vr

    left_df  = df[df[best_feat] <= best_thresh]
    right_df = df[df[best_feat] >  best_thresh]

    node.children['left']  = build_tree(left_df,  features, target, max_depth, min_samples, depth + 1)
    node.children['right'] = build_tree(right_df, features, target, max_depth, min_samples, depth + 1)

    return node


features = ['size_sqft', 'bedrooms', 'age_years', 'distance_km']
tree = build_tree(df, features, 'price_k', max_depth=4, min_samples=3)

print('Tree built!')
print('Root node:', tree)

Tree built!
Root node: Node(split: size_sqft <= 1750.0, n=25)


---
## Step 6 — Visualize the Tree

In [7]:
def print_tree(node, indent=0, branch='ROOT'):
    pad = '│   ' * indent
    if node.is_leaf:
        print(f'{pad}└─ [{branch}]')
        print(f'{pad}   → PREDICT: ${node.prediction:.1f}k  '
              f'(mean=${node.mean:.1f}k, n={node.n_samples})')
    else:
        print(f'{pad}└─ [{branch}]')
        print(f'{pad}   Split ⟨{node.feature} <= {node.threshold}⟩  '
              f'VR={node.var_reduction:.1f}  n={node.n_samples}')
        print_tree(node.children['left'],  indent + 1, f'{node.feature} <= {node.threshold}')
        print_tree(node.children['right'], indent + 1, f'{node.feature} >  {node.threshold}')

print('REGRESSION TREE')
print('=' * 65)
print_tree(tree)

REGRESSION TREE
└─ [ROOT]
   Split ⟨size_sqft <= 1750.0⟩  VR=6497.6  n=25
│   └─ [size_sqft <= 1750.0]
│      Split ⟨size_sqft <= 1325.0⟩  VR=1824.2  n=17
│   │   └─ [size_sqft <= 1325.0]
│   │      Split ⟨size_sqft <= 1015.0⟩  VR=698.8  n=9
│   │   │   └─ [size_sqft <= 1015.0]
│   │   │      Split ⟨size_sqft <= 860.0⟩  VR=182.2  n=4
│   │   │   │   └─ [size_sqft <= 860.0]
│   │   │   │      → PREDICT: $151.5k  (mean=$151.5k, n=2)
│   │   │   │   └─ [size_sqft >  860.0]
│   │   │   │      → PREDICT: $178.5k  (mean=$178.5k, n=2)
│   │   │   └─ [size_sqft >  1015.0]
│   │   │      Split ⟨size_sqft <= 1125.0⟩  VR=134.4  n=5
│   │   │   │   └─ [size_sqft <= 1125.0]
│   │   │   │      → PREDICT: $204.0k  (mean=$204.0k, n=2)
│   │   │   │   └─ [size_sqft >  1125.0]
│   │   │   │      → PREDICT: $227.7k  (mean=$227.7k, n=3)
│   │   └─ [size_sqft >  1325.0]
│   │      Split ⟨size_sqft <= 1525.0⟩  VR=570.0  n=8
│   │   │   └─ [size_sqft <= 1525.0]
│   │   │      Split ⟨age_years <= 5.0⟩  VR=126

---
## Step 7 — Predict on New Samples

Walk the tree: at each internal node, check the feature value against the threshold,
go left if `<=`, right if `>`. Stop at a leaf and return its `prediction` (the mean of its training samples).

In [8]:
def predict(node, sample):
    """Walk the tree to predict the price for one sample (dict or pd.Series)."""
    if node.is_leaf:
        return node.prediction

    if sample[node.feature] <= node.threshold:
        return predict(node.children['left'], sample)
    else:
        return predict(node.children['right'], sample)


# New houses not in training data
new_houses = pd.DataFrame([
    {'size_sqft': 950,  'bedrooms': 2, 'age_years': 12, 'distance_km': 8},
    {'size_sqft': 1350, 'bedrooms': 3, 'age_years': 5,  'distance_km': 4},
    {'size_sqft': 1700, 'bedrooms': 4, 'age_years': 2,  'distance_km': 3},
    {'size_sqft': 2200, 'bedrooms': 5, 'age_years': 1,  'distance_km': 1},
    {'size_sqft': 2600, 'bedrooms': 6, 'age_years': 4,  'distance_km': 2},
])

new_houses['predicted_price_k'] = new_houses.apply(lambda r: predict(tree, r), axis=1)
new_houses['predicted_price_$'] = (new_houses['predicted_price_k'] * 1000).map('${:,.0f}'.format)

print('Predictions on new houses:')
new_houses

Predictions on new houses:


,size_sqft,bedrooms,age_years,distance_km,predicted_price_k,predicted_price_$
0,950,2,12,8,178.5,"$178,500"
1,1350,3,5,4,267.5,"$267,500"
2,1700,4,2,3,294.0,"$294,000"
3,2200,5,1,1,409.0,"$409,000"
4,2600,6,4,2,482.5,"$482,500"


---
## Step 8 — Evaluate: MSE, MAE, RMSE

Regression uses different metrics than classification (no accuracy/F1):

| Metric | Formula | Meaning |
|---|---|---|
| **MSE** | mean of (actual − predicted)² | Penalises large errors heavily |
| **RMSE** | √MSE | Same units as target (dollars) |
| **MAE** | mean of |actual − predicted| | Average absolute error |
| **R²** | 1 − MSE/Var(y) | % of variance explained (1.0 = perfect) |

In [9]:
df['predicted'] = df.apply(lambda r: predict(tree, r), axis=1)
df['error']     = df['price_k'] - df['predicted']
df['abs_error'] = df['error'].abs()

mse  = (df['error'] ** 2).mean()
rmse = math.sqrt(mse)
mae  = df['abs_error'].mean()
r2   = 1 - mse / variance(df['price_k'])

print('Training set evaluation:')
print(f'  MSE  : {mse:.2f}')
print(f'  RMSE : {rmse:.2f}k  (avg error ≈ ${rmse*1000:,.0f})')
print(f'  MAE  : {mae:.2f}k  (avg abs error ≈ ${mae*1000:,.0f})')
print(f'  R²   : {r2:.4f}  ({r2*100:.1f}% of variance explained)')

print()
print('Full prediction table:')
print('-' * 70)
result = df[['size_sqft','bedrooms','price_k','predicted','error']].copy()
result['error'] = result['error'].map(lambda x: f'{x:+.1f}k')
result

Training set evaluation:
  MSE  : 106.21
  RMSE : 10.31k  (avg error ≈ $10,306)
  MAE  : 8.05k  (avg abs error ≈ $8,054)
  R²   : 0.9883  (98.8% of variance explained)

Full prediction table:
----------------------------------------------------------------------


,size_sqft,bedrooms,price_k,predicted,error
0,750,1,145,151.50,-6.5k
1,820,1,158,151.50,+6.5k
2,900,2,172,178.50,-6.5k
3,980,2,185,178.50,+6.5k
4,1050,2,198,204.00,-6.0k
5,1100,2,210,204.00,+6.0k
6,1150,3,225,227.67,-2.7k
7,1200,3,238,227.67,+10.3k
8,1300,3,220,227.67,-7.7k
9,1350,3,260,267.50,-7.5k


---
## Step 9 — Feature Importance (from the tree, not raw VR)

Same correct approach as the classification notebook — walk every internal node and accumulate:

$$importance[f] += \frac{n_{node}}{n_{total}} \times VR_{node}$$

Nodes higher up (more samples) contribute more weight.
Nodes deep in a small branch contribute less.

In [10]:
def feature_importance(tree, total_samples):
    scores = defaultdict(float)

    def walk(node):
        if node.is_leaf:
            return
        weight = node.n_samples / total_samples
        scores[node.feature] += weight * node.var_reduction
        for child in node.children.values():
            walk(child)

    walk(tree)

    # Normalize so importances sum to 1
    total = sum(scores.values())
    return {f: round(v / total, 4) for f, v in scores.items()}


importance = feature_importance(tree, total_samples=len(df))

imp_df = pd.DataFrame(importance.items(), columns=['feature','importance']) \
           .sort_values('importance', ascending=False).reset_index(drop=True)

print('Feature importance (from tree structure, normalized):')
print()
for _, row in imp_df.iterrows():
    bar = '█' * int(row['importance'] * 50)
    print(f"  {row['feature']:<15} {row['importance']:.4f}  {bar}")

print()
print(f"  Total: {imp_df['importance'].sum():.4f}  (should be 1.0)")

Feature importance (from tree structure, normalized):

  size_sqft       0.9960  █████████████████████████████████████████████████
  age_years       0.0023  
  distance_km     0.0018  

  Total: 1.0001  (should be 1.0)


---
## Step 10 — Effect of max_depth (Overfitting vs Underfitting)

This is the most important hyperparameter for regression trees.

- **Too shallow (depth=1):** Tree can't capture patterns — high error (underfitting)
- **Too deep (depth=10):** Tree memorises training data — zero training error but bad on new data (overfitting)
- **Sweet spot:** Minimise error on unseen data

In [11]:
print(f'{"Depth":<8} {"RMSE (train)":<18} {"MAE (train)":<15} {"R²"}')
print('-' * 55)

for depth in range(1, 8):
    t = build_tree(df, features, 'price_k', max_depth=depth, min_samples=2)
    preds  = df.apply(lambda r: predict(t, r), axis=1)
    errors = df['price_k'] - preds
    mse_d  = (errors**2).mean()
    rmse_d = math.sqrt(mse_d)
    mae_d  = errors.abs().mean()
    r2_d   = 1 - mse_d / variance(df['price_k'])
    note   = '← underfitting' if depth <= 1 else ('← overfitting' if depth >= 6 else '')
    print(f'{depth:<8} {rmse_d:<18.2f} {mae_d:<15.2f} {r2_d:.4f}  {note}')

Depth    RMSE (train)       MAE (train)     R²
-------------------------------------------------------
1        50.85              42.22           0.7153  ← underfitting
2        27.34              24.10           0.9177  
3        13.04              10.79           0.9813  
4        6.00               4.97            0.9960  
5        1.22               0.48            0.9998  
6        0.00               0.00            1.0000  ← overfitting
7        0.00               0.00            1.0000  ← overfitting


---
## Step 11 — Tree Statistics

In [12]:
def tree_stats(node):
    if node.is_leaf:
        return {'depth': 0, 'nodes': 1, 'leaves': 1}
    child_stats = [tree_stats(c) for c in node.children.values()]
    return {
        'depth' : 1 + max(s['depth']  for s in child_stats),
        'nodes' : 1 + sum(s['nodes']  for s in child_stats),
        'leaves': sum(s['leaves'] for s in child_stats),
    }

stats = tree_stats(tree)
print(pd.DataFrame([{
    'Max depth'      : stats['depth'],
    'Total nodes'    : stats['nodes'],
    'Leaf nodes'     : stats['leaves'],
    'Internal nodes' : stats['nodes'] - stats['leaves'],
}]).to_string(index=False))

 Max depth  Total nodes  Leaf nodes  Internal nodes
         4           23          12              11


---
## Summary

### Classification vs Regression tree — what changes

| | Classification | Regression |
|---|---|---|
| **Leaf output** | Majority class label | Mean of target values |
| **Impurity measure** | Entropy / Gini | Variance |
| **Split criterion** | Information Gain | Variance Reduction |
| **Feature splits** | By category value | By threshold on numeric value |
| **Evaluation** | Accuracy, F1 | MSE, RMSE, MAE, R² |
| **Overfitting risk** | Moderate | High — `max_depth` is critical |

### What this tree does NOT do (yet)
- **No train/test split** — all evaluation is on training data
- **No cross-validation** — proper depth selection needs held-out data
- **No pruning** — post-pruning (cost-complexity) can improve generalisation
- **No handling of missing values**

These are addressed by **CART** (the algorithm sklearn uses) and ensemble methods like **Random Forest** and **Gradient Boosting** which build many trees and combine their predictions.